In this notebook I consume Jira's REST API to extract issues. The process includes the follow ELT(L) process:
- E - Extract issues from Jira
- L - Save data to a JSON file
- T - Transforms data keeping only the required fields
- L - Store the transformed data into a new JSON file 

In [7]:
import math
import os
from pathlib import Path

import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col

In [ ]:
print(os.environ['jira_base64_auth_token'])

# EXTRACT + LOAD

In [16]:
max_results_per_page = 100
fields_to_return = ("assignee,reporter,customfield_10710,issuetype,project,resolutiondate,customfield_10709,"
                    "customfield_10016,updated,customfield_10015,summary,duedate,customfield_10590,customfield_10591,"
                    "customfield_10588,customfield_10589,customfield_10469,priority,customfield_10578,status,creator,creator,created")
jql = "project=DAK"

total_pages = 1
current_page = 0

headers = {
    'Accept': 'application/json',
    'Authorization': os.environ['jira_base64_auth_token']
}

while current_page < total_pages:
    start_at = current_page * max_results_per_page
    url = (f"https://simplesdental.atlassian.net/rest/api/3/search?expand="
           f"&maxResults={max_results_per_page}"
           f"&fields={fields_to_return}"
           f"&jql={jql}"
           f"&startAt={start_at}")

    response = requests.request("GET", url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Erro na requisicao da API: {response.status_code} - {response.text}")

    # So precisamos calcular o maximo de paginas na primeira interacao
    if current_page == 0:
        json_response = response.json()
        total_pages = math.ceil(json_response['total'] / max_results_per_page)

    current_page += 1

    response_text = response.text
    file_path = f"{Path().absolute()}/jira_issues_json_{current_page}.json"

    with open(file_path, 'w', encoding="utf-8") as json_file:
        json_file.write(response_text)

    print(f"Carregado pagina {current_page} de {total_pages}")


Carregado pagina 1 de 5
Carregado pagina 2 de 5
Carregado pagina 3 de 5
Carregado pagina 4 de 5
Carregado pagina 5 de 5


In [6]:
json_response = response.json()

print(json_response['maxResults'])
print(json_response['total'])

1
442


# Transform 

In [3]:
spark = SparkSession.builder.appName("jira_extract_issues").getOrCreate()

Create a new list of issues keeping only the required fields and renaming custom ones

In [4]:
file_path = f"{Path().absolute()}/jira_issues_json.json"
df = spark.read.json(file_path)

In [5]:
# df.printSchema()
df.show()

+------------+--------------------+----------+--------------------+-------+-----+
|      expand|              issues|maxResults|               names|startAt|total|
+------------+--------------------+----------+--------------------+-------+-----+
|schema,names|[{operations,vers...|        10|{Σ Progress, Σ Re...|      0|  434|
+------------+--------------------+----------+--------------------+-------+-----+



In [6]:
df_issues = df.select("issues")

In [7]:
# df_issues.show()
df_issues.printSchema()

root
 |-- issues: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- expand: string (nullable = true)
 |    |    |-- fields: struct (nullable = true)
 |    |    |    |-- aggregateprogress: struct (nullable = true)
 |    |    |    |    |-- progress: long (nullable = true)
 |    |    |    |    |-- total: long (nullable = true)
 |    |    |    |-- aggregatetimeestimate: string (nullable = true)
 |    |    |    |-- aggregatetimeoriginalestimate: string (nullable = true)
 |    |    |    |-- aggregatetimespent: string (nullable = true)
 |    |    |    |-- assignee: struct (nullable = true)
 |    |    |    |    |-- accountId: string (nullable = true)
 |    |    |    |    |-- accountType: string (nullable = true)
 |    |    |    |    |-- active: boolean (nullable = true)
 |    |    |    |    |-- avatarUrls: struct (nullable = true)
 |    |    |    |    |    |-- 16x16: string (nullable = true)
 |    |    |    |    |    |-- 24x24: string (nullable = true)
 |  

In [10]:
df_issues_flat = df_issues.withColumn("issues_exploded", explode("issues")) \
    .drop("issues")

In [11]:
# df_issues_flat.printSchema()
df_issues_flat.show(truncate=False)
# df_issues_flat.toPandas()

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
columns_to_keep = ["issues_exploded.id",
                   "issues_exploded.key",
                   "issues_exploded.fields.lastViewed",
                   col("issues_exploded.fields.assignee.accountId").alias("assignee_id"),
                   col("issues_exploded.fields.assignee.displayName").alias("assignee_name"),
                   col("issues_exploded.fields.reporter.accountId").alias("reporter_id"),
                   col("issues_exploded.fields.reporter.displayName").alias("reporter_name"),
                   col("issues_exploded.fields.customfield_10710.value").alias("pilar_de_atuacao"),
                   col("issues_exploded.fields.issuetype.name").alias("issue_type"),
                   col("issues_exploded.fields.project.id").alias("project_id"),
                   col("issues_exploded.fields.project.key").alias("project_key"),
                   col("issues_exploded.fields.project.name").alias("project_name"),
                   col("issues_exploded.fields.resolutiondate").alias("resolution_date"),
                   col("issues_exploded.fields.customfield_10709.value").alias("time_solicitante"),
                   col("issues_exploded.fields.customfield_10016").alias("story_point_estimate"),
                   col("issues_exploded.fields.updated"),
                   col("issues_exploded.fields.customfield_10015").alias("start_date"),
                   col("issues_exploded.fields.summary"),
                   col("issues_exploded.fields.duedate").alias("due_date"),
                   col("issues_exploded.fields.customfield_10590").alias("to_release_date"),
                   col("issues_exploded.fields.customfield_10591").alias("blocked_date"),
                   col("issues_exploded.fields.customfield_10588").alias("in_progress_date"),
                   col("issues_exploded.fields.customfield_10589").alias("to_review_date"),
                   col("issues_exploded.fields.customfield_10469.displayName").alias("solicitantes"),
                   col("issues_exploded.fields.priority.name").alias("priority"),
                   col("issues_exploded.fields.customfield_10578").alias("data_da_solicitacao"),
                   col("issues_exploded.fields.status.name").alias("status"),
                   col("issues_exploded.fields.creator.accountId").alias("creator_id"),
                   col("issues_exploded.fields.creator.displayName").alias("creator_name"),
                   col("issues_exploded.fields.created").alias("created"),
                   ]

df_issues_flat_clean = df_issues_flat.select(*columns_to_keep)

In [18]:
df_issues_flat_clean.printSchema()
# df_issues_flat_clean.show()

root
 |-- id: string (nullable = true)
 |-- key: string (nullable = true)
 |-- lastViewed: string (nullable = true)
 |-- assignee_id: string (nullable = true)
 |-- assignee_name: string (nullable = true)
 |-- reporter_id: string (nullable = true)
 |-- reporter_name: string (nullable = true)
 |-- pilar_de_atuacao: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- project_id: string (nullable = true)
 |-- project_key: string (nullable = true)
 |-- project_name: string (nullable = true)
 |-- resolution_date: string (nullable = true)
 |-- time_solicitante: string (nullable = true)
 |-- story_point_estimate: double (nullable = true)
 |-- updated: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- due_date: string (nullable = true)
 |-- to_release_date: string (nullable = true)
 |-- blocked_date: string (nullable = true)
 |-- in_progress_date: string (nullable = true)
 |-- to_review_date: string (nullable = tru

# Load

In [19]:
output_path = f"{Path().absolute()}/jira_issues_json_transformed.json"
json_array = df_issues_flat_clean.toJSON().collect()

with open(output_path, 'w', encoding="utf-8") as json_file:
    json_file.writelines(json_array)